In [1]:
import cv2
import mediapipe as mp
import pyautogui
import random
import util
import numpy as np
import time
import os
from datetime import datetime
from pynput.mouse import Button, Controller

# ==========================================
# INITIALIZATION & GLOBALS
# ==========================================
mouse = Controller()
screen_width, screen_height = pyautogui.size()

# Mouse Smoothing & Screen Edge Variables
plocX, plocY = 0, 0
clocX, clocY = 0, 0
smoothening = 5
margin = 0.15  # 15% margin to help reach screen edges

mpHands = mp.solutions.hands
hands = mpHands.Hands(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
    max_num_hands=1
)

# ==========================================
# 1. BASIC UTILITIES
# ==========================================
def find_finger_tip(processed):
    if processed.multi_hand_landmarks:
        hand_landmarks = processed.multi_hand_landmarks[0]
        return hand_landmarks.landmark[mpHands.HandLandmark.INDEX_FINGER_TIP]
    return None

def move_mouse(index_finger_tip):
    """Move cursor smoothly and allow it to reach screen edges easily."""
    global plocX, plocY, clocX, clocY
    
    if index_finger_tip is not None:
        # Apply margin so we don't have to move off-camera to hit the screen edges
        adj_x = (index_finger_tip.x - margin) / (1.0 - 2 * margin)
        adj_y = (index_finger_tip.y - margin) / (1.0 - 2 * margin)
        
        # Map to screen dimensions
        x = np.clip(int(adj_x * screen_width), 0, screen_width - 1)
        y = np.clip(int(adj_y * screen_height), 0, screen_height - 1)
        
        # Smooth the movement
        clocX = plocX + (x - plocX) / smoothening
        clocY = plocY + (y - plocY) / smoothening
        
        pyautogui.moveTo(clocX, clocY)
        plocX, plocY = clocX, clocY

# ==========================================
# 2. GESTURE CONDITIONS (Using Pixel Math)
# ==========================================
def is_left_click(landmark_list, thumb_index_dist):
    return (
        util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) < 50 and
        util.get_angle(landmark_list[9], landmark_list[10], landmark_list[12]) > 90 and
        thumb_index_dist > 50
    )

def is_right_click(landmark_list, thumb_index_dist):
    return (
        util.get_angle(landmark_list[9], landmark_list[10], landmark_list[12]) < 50 and
        util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) > 90 and
        thumb_index_dist > 50
    )

def is_double_click(landmark_list):
    index_angle = util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8])
    middle_angle = util.get_angle(landmark_list[9], landmark_list[10], landmark_list[12])
    thumb_index_dist = util.get_distance([landmark_list[4], landmark_list[8]])
    return index_angle < 50 and middle_angle < 50 and thumb_index_dist > 100

def is_fist_closed(landmark_list):
    tip_ids = [8, 12, 16, 20]
    return all(landmark_list[i][1] > landmark_list[i - 2][1] for i in tip_ids)

def is_palm_open(landmark_list):
    tip_ids = [4, 8, 12, 16, 20]
    fingers = [landmark_list[tip_ids[i]][1] < landmark_list[tip_ids[i] - 2][1] for i in range(1, 5)]
    return all(fingers)

def is_volume_up(landmark_list):
    dist = util.get_distance([landmark_list[4], landmark_list[8]])
    return dist > 150 # Now works because we use pixel coordinates!

def is_volume_down(landmark_list):
    dist = util.get_distance([landmark_list[4], landmark_list[8]])
    return dist < 40  # Now works because we use pixel coordinates!

def is_thumb_open(landmark_list):
    thumb_tip = landmark_list[4]
    thumb_mcp = landmark_list[2]
    index_mcp = landmark_list[5]
    # ~45 pixels distance threshold
    return abs(thumb_tip[0] - thumb_mcp[0]) > 45 and abs(thumb_tip[0] - index_mcp[0]) > 45

def are_two_fingers_up(landmark_list):
    fingers = []
    fingers.append(landmark_list[8][1] < landmark_list[6][1])   # index up
    fingers.append(landmark_list[12][1] < landmark_list[10][1]) # middle up
    fingers.append(landmark_list[16][1] < landmark_list[14][1]) # ring down
    fingers.append(landmark_list[20][1] < landmark_list[18][1]) # pinky down
    return fingers[0] and fingers[1] and not fingers[2] and not fingers[3]

def is_screenshot(landmark_list):
    # ~10 pixel safety margin
    def finger_up(tip_id, pip_id):
        return landmark_list[tip_id][1] < landmark_list[pip_id][1] - 10

    def finger_down(tip_id, pip_id):
        return landmark_list[tip_id][1] > landmark_list[pip_id][1] + 10

    index_up = finger_up(8, 6)
    middle_up = finger_up(12, 10)
    ring_up = finger_up(16, 14)
    pinky_down = finger_down(20, 18)
    thumb_down = landmark_list[4][0] < landmark_list[3][0]

    return index_up and middle_up and ring_up and thumb_down and pinky_down

# ==========================================
# 3. ACTIONS
# ==========================================
def take_screenshot(frame):
    folder = os.path.expanduser("~/Pictures/Screenshots")
    os.makedirs(folder, exist_ok=True)
    filename = f"Screenshot_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.png"
    path = os.path.join(folder, filename)
    im1 = pyautogui.screenshot()
    im1.save(path)
    print(f"Screenshot saved at: {path}")

# ==========================================
# 4. GESTURE HANDLER (With Anti-Spam)
# ==========================================
last_action_time = 0
COOLDOWN = 0.6
last_play_state = False
last_pause_state = False
current_gesture = None
gesture_text = ""

def detect_gesture(frame, landmark_list, processed):
    global last_action_time, last_play_state, last_pause_state
    global current_gesture, gesture_text

    if len(landmark_list) < 21:
        current_gesture = None
        gesture_text = ""
        return

    index_finger_tip = find_finger_tip(processed)
    thumb_index_dist = util.get_distance([landmark_list[4], landmark_list[5]])

    # --- Mouse Movement ---
    if (
        are_two_fingers_up(landmark_list)
        and not is_thumb_open(landmark_list)
        and not is_palm_open(landmark_list)
        and not is_fist_closed(landmark_list)
    ):
        move_mouse(index_finger_tip)

    # --- Gesture Detection ---
    new_gesture = None
    if is_palm_open(landmark_list):
        new_gesture = "play"
    elif is_fist_closed(landmark_list):
        new_gesture = "pause"
    elif is_left_click(landmark_list, thumb_index_dist):
        new_gesture = "left_click"
    elif is_right_click(landmark_list, thumb_index_dist):
        new_gesture = "right_click"
    elif is_double_click(landmark_list):
        new_gesture = "double_click"
    elif is_screenshot(landmark_list):
        new_gesture = "screenshot"
    elif is_volume_up(landmark_list):
        new_gesture = "volume_up"
    elif is_volume_down(landmark_list):
        new_gesture = "volume_down"

    if new_gesture != current_gesture:
        current_gesture = new_gesture

    # --- Execute Action (with Cooldown) ---
    if current_gesture and (time.time() - last_action_time > COOLDOWN):
        
        if current_gesture == "play" and not last_play_state:
            pyautogui.press("playpause")
            gesture_text = "▶️ Play"
            last_play_state = True
            last_pause_state = False
            last_action_time = time.time()

        elif current_gesture == "pause" and not last_pause_state:
            pyautogui.press("playpause")
            gesture_text = "⏸️ Pause"
            last_pause_state = True
            last_play_state = False
            last_action_time = time.time()

        elif current_gesture == "left_click":
            mouse.click(Button.left)
            gesture_text = "🖱️ Left Click"
            last_action_time = time.time()

        elif current_gesture == "right_click":
            mouse.click(Button.right)
            gesture_text = "🖱️ Right Click"
            last_action_time = time.time()

        elif current_gesture == "double_click":
            pyautogui.doubleClick()
            gesture_text = "🖱️ Double Click"
            last_action_time = time.time()

        elif current_gesture == "screenshot":
            take_screenshot(frame)
            gesture_text = "📸 Screenshot"
            last_action_time = time.time()

        # Volume allows slightly faster execution (0.2s cooldown) so you can hold it
        elif current_gesture == "volume_up" and (time.time() - last_action_time > 0.2):
            pyautogui.press("volumeup", presses=2)
            gesture_text = "🔊 Volume Up"
            last_action_time = time.time()

        elif current_gesture == "volume_down" and (time.time() - last_action_time > 0.2):
            pyautogui.press("volumedown", presses=2)
            gesture_text = "🔉 Volume Down"
            last_action_time = time.time()

    # --- Display Info ---
    if current_gesture:
        cv2.putText(frame, gesture_text, (40, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
        
    # Draw invisible margin box on screen so you know your tracking boundaries
    h, w, c = frame.shape
    cv2.rectangle(frame, (int(w*margin), int(h*margin)), (int(w*(1-margin)), int(h*(1-margin))), (0, 255, 0), 1)

# ==========================================
# 5. MAIN CAMERA LOOP
# ==========================================
def main():
    draw = mp.solutions.drawing_utils
    cap = cv2.VideoCapture(0)

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            frame = cv2.flip(frame, 1)
            frameRGB = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            processed = hands.process(frameRGB)

            # Get actual frame dimensions for pixel math
            h, w, c = frame.shape
            landmark_list = []
            
            if processed.multi_hand_landmarks:
                hand_landmarks = processed.multi_hand_landmarks[0]
                draw.draw_landmarks(frame, hand_landmarks, mpHands.HAND_CONNECTIONS)
                
                # CRITICAL FIX: Convert normalized coordinates to pixel coordinates
                for lm in hand_landmarks.landmark:
                    landmark_list.append((int(lm.x * w), int(lm.y * h)))

            # Run detection
            detect_gesture(frame, landmark_list, processed)

            cv2.imshow('MANUMOTION Control Engine', frame)
            if cv2.waitKey(1) & 0xFF == 27: # Press ESC to close
                break

    finally:
        cap.release()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    main()